# Fine-tune BERT on the LIAR dataset for TruthSeeker

This notebook fine-tunes `bert-base-uncased` for binary fake/real claim classification on the LIAR dataset (Wang, 2017 — *"Liar, Liar Pants on Fire": A New Benchmark Dataset for Fake News Detection*), then pushes the result to your Hugging Face Hub account so it can be hosted for the TruthSeeker backend.

**Before running**: in Colab, go to Runtime -> Change runtime type -> select a GPU (T4 is fine, free tier). Training on CPU here will be far too slow.

**You will need**: a free Hugging Face account and an access token with write permission (huggingface.co/settings/tokens) for the last step.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

## 2. Download and extract the LIAR dataset

This pulls the original release from the paper's authors. If this specific URL is ever unreachable, search "LIAR dataset fake news benchmark" on Kaggle for a mirror with the same `train.tsv` / `valid.tsv` / `test.tsv` files, upload them to this Colab session, and skip this cell.

In [ ]:
import os, urllib.request, zipfile

url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
urllib.request.urlretrieve(url, "liar_dataset.zip")

with zipfile.ZipFile("liar_dataset.zip", "r") as z:
    z.extractall("liar_dataset")

os.listdir("liar_dataset")

## 3. Load the TSVs

LIAR's files have no header row. Columns per the dataset's README: id, label, statement, subject, speaker, job_title, state, party, barely_true_counts, false_counts, half_true_counts, mostly_true_counts, pants_on_fire_counts, context.

In [ ]:
import pandas as pd

columns = [
    "id", "label", "statement", "subject", "speaker", "job_title", "state", "party",
    "barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts",
    "pants_on_fire_counts", "context",
]

train_df = pd.read_csv("liar_dataset/train.tsv", sep="\t", header=None, names=columns)
valid_df = pd.read_csv("liar_dataset/valid.tsv", sep="\t", header=None, names=columns)
test_df = pd.read_csv("liar_dataset/test.tsv", sep="\t", header=None, names=columns)

print(len(train_df), len(valid_df), len(test_df))
train_df[["label", "statement"]].head()

## 4. Collapse the 6-way label to binary

LIAR's original labels are a 6-point truthfulness scale: pants-fire, false, barely-true, half-true, mostly-true, true. TruthSeeker's `bert_api.py` expects a binary fake/real classifier, so we collapse the scale at its midpoint. This is a deliberate simplification worth mentioning in your project write-up: it trades some nuance for a decision the rest of the pipeline can act on directly.

In [ ]:
FAKE_LABELS = {"pants-fire", "false", "barely-true"}
REAL_LABELS = {"half-true", "mostly-true", "true"}

def to_binary(label):
    label = str(label).strip().lower()
    if label in FAKE_LABELS:
        return 0  # fake
    if label in REAL_LABELS:
        return 1  # real
    return None

for df in (train_df, valid_df, test_df):
    df["binary_label"] = df["label"].apply(to_binary)
    df.dropna(subset=["binary_label"], inplace=True)
    df["binary_label"] = df["binary_label"].astype(int)

train_df["binary_label"].value_counts()

## 5. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    renamed = df[["statement", "binary_label"]].rename(
        columns={"statement": "text", "binary_label": "label"}
    )
    return Dataset.from_pandas(renamed, preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = to_hf_dataset(train_df).map(tokenize_fn, batched=True)
valid_ds = to_hf_dataset(valid_df).map(tokenize_fn, batched=True)
test_ds = to_hf_dataset(test_df).map(tokenize_fn, batched=True)

## 6. Fine-tune

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./bert-liar-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
)

trainer.train()

## 7. Evaluate on the held-out test split

This is the real accuracy number for your project write-up — report this, not a training-set number.

In [ ]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("Accuracy:", accuracy_metric.compute(predictions=preds, references=labels))
print("F1:", f1_metric.compute(predictions=preds, references=labels))

## 8. Save locally and push to your Hugging Face Hub account

Running the login cell will prompt for a token — create one (write access) at huggingface.co/settings/tokens and paste it in. Change `YOUR_HF_USERNAME` below to your actual Hugging Face username.

In [ ]:
model.save_pretrained("bert-liar-model")
tokenizer.save_pretrained("bert-liar-model")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
HF_REPO_ID = "YOUR_HF_USERNAME/truthseeker-bert-liar"  # change this

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)

print(f"Pushed. Your model is now at https://huggingface.co/{HF_REPO_ID}")
print(f"Set BERT_MODEL_ID={HF_REPO_ID} on the hosted BERT service.")